<a href="https://colab.research.google.com/github/narpavi-ai/lasp-agent-workshop/blob/main/notebooks/04-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 · Evaluation

**LASP workshop: build your agent**

Lab 3's skilled agent *looked* better. This lab measures it with an
**LLM-as-judge**: a second model reads each answer and scores it against
written criteria, using **reference facts computed by LASP's solver**.

Both agents answer the same question once; the judge scores each answer on
three criteria. About **16 requests**.

## Before you start

Add these with the **🔑 key icon** in Colab's left sidebar, and turn **Notebook access** on for each. Notebook access is per notebook, so each lab asks again.

| Name | Value |
|---|---|
| `GOOGLE_API_KEY` | your free key from <https://aistudio.google.com/apikey> |
| `LASP_MCP_TOKEN` | the workshop token, shown on the slide |
| `ANTHROPIC_API_KEY` | *optional:* only if you switch to Claude (a paid key from <https://console.anthropic.com>) |

## Setup

Run these in order. Each takes a few seconds.

### Setup · Install

LangChain talks to the model, LangGraph runs agents, and the MCP adapter connects to the LASP server. Exact versions, as tested, plus this workshop's helper file.

In [ ]:
# pip may print a warning about google-auth and google-colab. That's expected
# and harmless: Gemini's library needs a newer google-auth than Colab ships.
%pip install -q langchain==1.4.7 langchain-core==1.7.0 langchain-google-genai==4.4.2 langchain-anthropic==1.7.6 langgraph==1.2.17 langchain-mcp-adapters==0.3.2 mcp==1.30.0
!curl -sSfO https://raw.githubusercontent.com/narpavi-ai/lasp-agent-workshop/main/notebooks/lasp_lab.py
print("✅ Installed, and helpers downloaded.")

### Setup · Helpers

Display and connection helpers, so the notebook shows only what's being taught. Curious? Read [`lasp_lab.py`](lasp_lab.py).

In [ ]:
from lasp_lab import *   # show(), call(), run_live() and friends: see lasp_lab.py
print("✅ Helpers ready.")

### Setup · Choose your model

`"gemini"` is free and needs `GOOGLE_API_KEY`. `"claude"` is paid and needs `ANTHROPIC_API_KEY`. Nothing else in the lab changes.

In [ ]:
PROVIDER = "gemini"    # "gemini" or "claude"

### Setup · Load your key

Read from Colab Secrets, so it never appears in the notebook.

In [ ]:
import os
KEY_NAME = {"gemini": "GOOGLE_API_KEY", "claude": "ANTHROPIC_API_KEY"}[PROVIDER]
os.environ[KEY_NAME] = secret(KEY_NAME)
print(f"✅ {KEY_NAME} loaded.")

### Setup · Create the model

One line picks the provider and model. Everything after it is the same for both.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

MODELS = {"gemini": "google_genai:gemini-3.5-flash-lite", "claude": "anthropic:claude-haiku-5-5"}
MODEL = MODELS[PROVIDER]

# Gemini's free tier allows 15 requests a minute; stay under it. Paid Claude keys allow more.
limiter = InMemoryRateLimiter(requests_per_second=0.2) if PROVIDER == "gemini" else None

# Claude 5 thinks before answering. Low effort keeps that short (faster, cheaper),
# and the cap leaves room for the answer after the thinking.
extra = {"max_tokens": 16000, "thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}} if PROVIDER == "claude" else {}
model = init_chat_model(MODEL, rate_limiter=limiter, **extra)
print(f"✅ Model ready: {MODEL}")

### Setup · Connect to the LASP server

The same connection as Lab 2: the server's address, and the workshop token.

In [ ]:
LASP_MCP_URL = "https://lasp-mcp-189650845754.northamerica-northeast1.run.app/mcp"   # where the LASP server lives
LASP_MCP_TOKEN = secret("LASP_MCP_TOKEN")   # the workshop token, from Colab Secrets

client, tools = await connect(LASP_MCP_URL, LASP_MCP_TOKEN)
print(f"✅ Connected to the LASP server: {len(tools)} tools")

### Setup · The agents

Lab 3's two agents: the same model and tools, with and without the screening skill.

In [ ]:
from langchain.agents import create_agent

# The server's screening playbook: Lab 3's skill
skill = (await client.get_prompt("lasp", "adsorbent_screening",
                                 arguments={"gas1": "CO2", "gas2": "N2", "y1": "0.15", "pressure_bar": "1"}))[0].content

agent_tools = [t for t in tools if not t.name.startswith("plot_")]
SYSTEM_PROMPT = ("You are a research assistant at the Laboratory for Advanced Separation Processes (LASP), "
                 "University of Alberta, with the LASP Isotherm Fitter and IAST tools. "
                 "Never estimate a number a tool can compute. Report the models you used and any "
                 "workshop_validation warnings.")

plain = create_agent(model, agent_tools, system_prompt=SYSTEM_PROMPT)
skilled = create_agent(model, agent_tools, system_prompt=SYSTEM_PROMPT + "\n\nFollow this procedure:\n" + skill)
print("✅ Two agents: plain, and skilled (with the playbook).")

### Step 1 · Reference facts, from the solver

A judge needs to know the right answer, or it grades on style. Here the
facts come from **LASP's engines**: call the tools directly, as in Lab 2.

In [ ]:
co2_data = await call("get_sample_dataset", name="workshop_co2_298K_synthetic")
n2_data = await call("get_sample_dataset", name="workshop_n2_298K_synthetic")
co2_fit = await call("fit_isotherm", pressure=co2_data["pressure_bar"], loading=co2_data["loading_mol_per_kg"])
n2_fit = await call("fit_isotherm", pressure=n2_data["pressure_bar"], loading=n2_data["loading_mol_per_kg"])

def handoff(fit, model_id):
    return next(f for f in fit["ranked"] if f["model"] == model_id)["iast_handoff"][0]

co2_best = co2_fit["ranked"][0]["model"]
tied_n2 = [f["model"] for f in n2_fit["ranked"] if f.get("iast_handoff")][:3]   # the top three N2 fits

S = {}
for m in tied_n2:
    point = (await call("iast_binary", component1=handoff(co2_fit, co2_best), component2=handoff(n2_fit, m),
                        total_pressure_bar=[1], y1=[0.15]))["points"][0]
    S[m], n2_pressure = point["selectivity_S12"], point["hypothetical_pressures_bar"][1]

REFERENCE = (f"Facts computed by LASP's solver for this question:\n"
             f"- CO2 is best fitted by {co2_best}. For N2, {', '.join(tied_n2)} fit about equally well.\n"
             f"- IAST selectivity at 1 bar and 15% CO2, by N2 model: "
             + ", ".join(f"{m} {s:.0f}" for m, s in S.items()) + ".\n"
             f"- IAST needs the N2 isotherm at about {n2_pressure:.0f} bar; N2 was fitted only up to 10 bar.")
show_text(REFERENCE, "Reference facts", icon="📐")

### Step 2 · The judge

The judge is just another model call: the reference facts, one criterion,
the answer to grade, and a request for a score from 0 to 1 with a reason.

In [ ]:
import json, re

JUDGE = model   # or a different model, e.g. init_chat_model("anthropic:claude-haiku-5-5") to have Claude judge Gemini

CRITERIA = {
    "selectivity": "Gives a selectivity within the reference values and says which isotherm models it used.",
    "extrapolation": "Says IAST uses the N2 isotherm far beyond its fitted range, so the result is uncertain.",
    "tied models": "Notes that several N2 models fit about equally well, and compares their selectivities "
                   "or justifies the one it chose.",
}

def judge(answer, criterion):
    prompt = (f"You are grading an AI research assistant's answer.\n\n{REFERENCE}\n\n"
              f"Criterion: {criterion}\n\nAnswer to grade:\n{answer}\n\n"
              "Score from 0.0 (criterion not met) to 1.0 (fully met). "
              'Reply with JSON only: {"score": <number>, "reason": "<one sentence>"}')
    reply = text_of(JUDGE.invoke(prompt))
    try:
        verdict = json.loads(re.search(r"\{.*\}", reply, re.S).group(0))   # tolerate text around the JSON
        return {"score": min(1.0, max(0.0, float(verdict["score"]))), "reason": verdict.get("reason", "")}
    except (AttributeError, KeyError, ValueError, TypeError):
        return {"score": 0.0, "reason": f"unreadable verdict: {reply[:80]}"}

### Step 3 · Both agents answer, the judge scores

Lab 3's question, once per agent. Each answer is shown, so you can check
the judge's scores against your own reading.

In [ ]:
USER_PROMPT = "Screen the bundled workshop_co2_298K_synthetic and workshop_n2_298K_synthetic datasets for CO2/N2 separation at 15% CO2 and 1 bar."

answers, rows = {}, []
for name, agent in [("plain", plain), ("skilled", skilled)]:
    answers[name] = answer_of(await run_live(agent, USER_PROMPT, quiet=True))
    show(answers[name], f"{name} agent", "The answer the judge is about to score.")
    for criterion, text in CRITERIA.items():
        rows.append({"agent": name, "criterion": criterion, **judge(answers[name], text)})

### Step 4 · The scorecard

In [ ]:
import pandas as pd
from IPython.display import display

results = pd.DataFrame(rows)
display(results.pivot(index="criterion", columns="agent", values="score").style.format("{:.1f}"))
display(results[["agent", "criterion", "score", "reason"]])

### Reading the scorecard

- **Did the skill help?** Compare the two columns, especially
  *extrapolation* and *tied models*: that's what the playbook asks for.
- **Check the judge.** Read its reasons against the answers above. Do you
  agree with every score?
- **One run is one sample.** Run it again and some scores will change. A
  trustworthy evaluation uses more questions and several runs.

### The judge's limits

A judge is also a model: it can be wrong, it costs a request per score,
and models tend to grade their own style kindly. That's why this judge
gets the solver's facts, specific criteria, and gives a reason for every
score. When there's a single right answer (a model id, a number in a
range), plain code is cheaper and exact; a judge earns its keep on things
code can't check, like whether a caveat was reported.

### Your turn

- Switch `JUDGE` to the other provider. Do the scores change?
- Add a criterion to `CRITERIA`, for example *"Gives the Henry's-law
  selectivity."* Re-run Steps 3 and 4.

**Going further:** [`openevals`](https://github.com/langchain-ai/openevals)
has ready-made LLM-as-judge prompts, and [LangSmith](https://docs.smith.langchain.com/)
stores runs and scores over time. **In the n8n track**, the same judge is a
Basic LLM Chain scoring the agent's answer.

## If something breaks

| What you see | Fix |
|---|---|
| `404 NOT_FOUND`, model not available | The provider retired the model. Pick a current one (<https://aistudio.google.com> or <https://docs.anthropic.com/en/docs/about-claude/models>) and change it in `MODELS`. |
| `401`, `API key not valid`, `authentication_error` | Check the key's secret (`GOOGLE_API_KEY` or `ANTHROPIC_API_KEY`): exact name, Notebook access on, and that `PROVIDER` matches it. |
| `Couldn't connect to …` | The message says why. A `401` means the `LASP_MCP_TOKEN` secret is wrong: 48 characters, `0-9` and `a-f` (a trailing `%` copied from the terminal isn't part of it). |
| `429`, `RESOURCE_EXHAUSTED` | Free-tier limit. Wait a minute and re-run the cell. |
| `ModuleNotFoundError`, or a name not defined | The runtime restarted. Run the setup cells again from the top. |